# Importation des bibliothèques et règlage de l'affichage

In [ ]:
import numpy as np
from numpy.polynomial.legendre import leggauss
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Polygon
from mpl_toolkits.mplot3d import Axes3D
import scipy
from scipy.sparse import coo_matrix, csc_matrix
from scipy.sparse.linalg import splu,spsolve
from scipy.linalg import null_space
import meshio
import pygmsh
from numpy.typing import ArrayLike
from scipy.sparse.linalg import eigs
import builtins
import numpy as np
from typing import Callable
import random
from matplotlib.collections import PolyCollection
from matplotlib.tri import Triangulation
import sympy as sp
from numpy.typing import NDArray

import builtins

if getattr(builtins.print, "__name__", "") != "custom_print":
    _original_print = builtins.print

    def custom_print(*args, **kwargs):
        def colorize(arg):
            if isinstance(arg, (bool, np.bool_)):
                return f'\033[94m{arg}\033[0m' if arg else f'\033[91m{arg}\033[0m'
            return arg

        args = [colorize(arg) for arg in args]
        _original_print(*args, **kwargs)

    builtins.print = custom_print


import sys
sys.path.append(r"C:\Users\stordeux\Dropbox\COURS\ANALYSE_NUMERIQUE\ELEMENTS_FINIS\Code_hyperbo\BIB")
from mes_packages import *

: 

In [ ]:
mesh = create_mesh_circle_in_square(radius=0.1, square_size=0.3, mesh_size=0.025)
ordre =2
# Declaration d'une matrice sparse pour stocker la matrice de masse globale
Nloc = (ordre + 1) * (ordre + 2) // 2
Nglob = nombre_dof_DG(mesh, ordre)

In [ ]:
mesh = create_mesh_circle_in_square(radius=0.1, square_size=0.3, mesh_size=0.025)
# Recuperation de la géométrie du maillage
points = mesh.points[:, :2]  # On ne garde que les coordonnées x et y
triangles = np.asarray(mesh.cells_dict["triangle"]) # mesh.cells_dict["triangle"]
# Construction de la structure de voisinage
neighbors, neighbor_faces, edges_to_triangles = build_neighborhood_structure(triangles)
loctoglob_DG = build_loctoglob_DG(triangles, ordre)


=== Vérification de l'orientation des triangles ===
Nombre total de triangles: 251

Nombre de triangles corrigés: 0/251
Tous les triangles étaient déjà correctement orientés True


# Determination des degres de liberte globaux de la face $i$ du triangle $T$ de reference ielt

La fonction `iface_iglob` (dans `methode_DG.py`) renvoie l'indice global du $iloc\_face$-ieme DDL situe sur la face $iface$ du triangle $ielt$.

Entrees principales :
- $ielt$ : numero du triangle.
- $iface \in \{0,1,2\}$ : face opposee au sommet 0, 1 ou 2.
- $iloc\_face \in [0,\,ordre]$ : position du point sur la face.
- $ordre$ : ordre polynomial.
- `loctoglob_DG` : table locale -> globale.

Principe :
- On exprime le point de la face en coordonnees locales entieres $(m,n)$ sur le triangle de reference.
- Conversion $(m,n) \rightarrow iloc$ via `loc2D_to_loc1D`.
- Conversion $iloc \rightarrow iglob$ via `loctoglob_DG[ielt, iloc]`.

Parametrisation des faces :
- Face 0 (opposee a A0) : $m + n = ordre$, avec $m = ordre - iloc\_face$, $n = iloc\_face$.
- Face 1 (opposee a A1) : $m = 0$, $n = ordre - iloc\_face$.
- Face 2 (opposee a A2) : $n = 0$, $m = iloc\_face$.

# Générer en fonction de T, F et iloc_face les deux numéros globaux des ddl de la face F

## Description de `get_face_dof_pair`

Cette fonction retourne les deux indices globaux associes au meme point de face :
- $iglob_T$ pour le triangle courant $T$.
- $iglob_V$ pour le triangle voisin $V$ (ou $-1$ si la face est au bord).

Etapes principales :
1. Convertir la position $iloc\_face$ sur la face $F$ en coordonnees locales entieres $(m,n)$ sur le triangle de reference.
2. Convertir $(m,n)$ en indice local $iloc$ avec `loc2D_to_loc1D`.
3. Convertir $iloc$ en indice global DG via $iglob = Nloc * T + iloc$, avec $Nloc = \frac{(ordre+1)(ordre+2)}{2}$.
4. Si un voisin existe, appliquer la meme conversion sur la face correspondante $F_V$, en inversant l'orientation (le point $iloc\_face$ sur $T$ correspond a $ordre - iloc\_face$ sur $V$).

En resume, `get_face_dof_pair` relie deux DDL qui representent le meme point geometrique mais appartiennent a deux triangles differents en DG.

In [1]:

# Test de la fonction
print("=== Test de la fonction get_face_dof_pair ===\n")

# Tirage d'un triangle pour le test
ielt_test = None
for i in range(len(triangles)):
    if np.all(neighbors[i] >= 0):  # Triangle avec 3 voisins
        ielt_test = i
        break

if ielt_test is None:
    ielt_test = 0
    print("⚠️  Utilisation du triangle 0 (peut avoir des faces de bord)")

print(f"Triangle testé : T = {ielt_test}")
print(f"Voisins : {neighbors[ielt_test]}")
print(f"Faces voisines : {neighbor_faces[ielt_test]}")
print()

# Test pour chaque face
for F in range(3):
    print(f"--- Face {F} ---")
    V = neighbors[ielt_test, F]
    if V >= 0:
        F_V = neighbor_faces[ielt_test, F]
        print(f"  Voisin V = {V}, face de V = {F_V}")
    else:
        print(f"  Pas de voisin (face de bord)")
    
    # Pour ordre 1, il y a 2 points par face (iloc_face = 0 et 1)
    for iloc_face in range(ordre + 1):
        iglob_T, iglob_V = get_face_dof_pair(ielt_test, F, iloc_face, ordre, neighbors, neighbor_faces)
        print(f"  iloc_face={iloc_face}: iglob_T={iglob_T}, iglob_V={iglob_V}")
    print()

=== Test de la fonction get_face_dof_pair ===



NameError: name 'triangles' is not defined